# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [1]:
#import sys; sys.path.append("/home/jovyan/tools")
import sys, os
sys.path.append(os.path.abspath('')+"/../tools") # This will work on every machiene. (as long as the notebook is one layer deep from root)

from lakelib import * # Auto-installing packages is kind of flakey.

(installing confluent-kafka==2.11.0 ...)
(installing duckdb==1.5.6 ...)


## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [2]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261125-132815-502b61

Lake check — topic 'logs' (4 partitions)   2026-11-25 13:28:39
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 480,939)     480,939           0          0           0
        1           [0, 473,674)     473,674           0          0           0
        2           [0, 538,749)     538,749           0          0           0
        3           [0, 506,638)     506,638           0          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   2,000,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 111.3 MB, 27.83 MB pe

{'topic': 2000000,
 'lake': 2000000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 111.330808,
 'ok': True,
 'problems': []}

In [3]:
run_once()                        # the next run
check()

read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261125-132839-f40dc6

Lake check — topic 'logs' (4 partitions)   2026-11-25 13:28:47
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 480,939)     961,878     480,939          0           0
        1           [0, 473,674)     947,348     473,674          0           0
        2           [0, 538,749)   1,077,498     538,749          0           0
        3           [0, 506,638)   1,013,276     506,638          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   4,000,000   2,000,000          0           0

files under /home/jovyan/data/lake: 8 data files, 222.7 MB, 27.83 MB per file on average
check took 0.0 s

Exa

{'topic': 2000000,
 'lake': 4000000,
 'dup': 2000000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 222.661616,
 'ok': False,
 'problems': ['2,000,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Architecture A saves offsets after appending. Architecture B stages each batch and resumes from the published data.
Architecture C uses the same writer as A and removes duplicates when reading with `DEDUP_SQL`.

In [1]:
def run_once_state_file():
    new = read_new(load_state())
    write_files(new, LAKE)
    save_state(new.end)


def run_once_lake_truth():
    new = read_new(lake_position())
    publish(write_files(new, STAGING)) # This will move the new data into the lake after the write_files() has completed.
    # e.g.: If write_files() fails, the lake will not be updated and the next run will re-read the same data again.


def run_once_rem_dup_read():
    run_once_state_file() # incl. lake_sql=DEDUP_SQL -> This will remove duplicates from the read, but not from the lake, since the lake isn't read again.
    # e.g.: It will just read the data without duplicates, but the lake will still contain them. (This is probably not the best option, but shold be used in tandem with the run_once_lake_truth() as a precaution.)


run_once = run_once_lake_truth # This is the chosen implementation.

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [ ]:
# DEDUP_SQL = f"(SELECT DISTINCT ON (kafka_partition, kafka_offset) * FROM {LAKE_SQL})"

check_all(run_once, lake_sql=DEDUP_SQL)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,600,000           0        0           0  12.1s  PASS
2 new records arrive, run again         2,700,000           0        0           0   2.9s  PASS
3 nothing new, run again                2,700,000           0        0           0   0.1s  PASS
4 run stops WHILE writing: right after  2,732,110           0        0      67,890         PASS
  ... then the next run                 2,800,000           0        0           0   2.8s  PASS
5 run stops AFTER writing: right after  2,900,000           0        0           0         PASS
  ... then the next run                 2,900,000           0        0           0   1.5s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 2600000,
   'lake': 2600000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 121.627614,
   'ok': True,
   'problems': []},
  12.120233297348022),
 ('2 new records arrive, run again',
  {'topic': 2700000,
   'lake': 2700000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 122.76641,
   'ok': True,
   'problems': []},
  2.944634437561035),
 ('3 nothing new, run again',
  {'topic': 2700000,
   'lake': 2700000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 122.76641,
   'ok': True,
   'problems': []},
  0.05396294593811035),
 ('4 run stops WHILE writing: right after',
  {'topic': 2800000,
   'lake': 2732110,
   'dup': 0,
   'miss': 0,
   'lag': 67890,
   'files': 9,
   'mb': 123.122744,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2800000,
   'lake': 2800000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 13,
   'mb': 124.228835,
   'ok': True,
   'problems': []},
  